# 04 · Customer Profile Engine

`get_profile(customer_id)` returns a layered, explainable profile for any active customer:

| Layer | Content | Source |
| --- | --- | --- |
| L1 | account dates, market (descriptive only) | point-in-time features |
| L2 | purchasing behaviour with percentile context | `customer_features(cutoff)` + `PERCENT_RANK` |
| L3 | timing, rhythm and trends | `customer_features(cutoff)` |
| L4 | segment, 90-day purchase propensity and decile | Steps 5 and 6 |
| L5 | features raising / lowering the score | Step 7 (TreeSHAP) |

Everything is read from one SQL view, `marts.customer_profile_base` (`sql/06_profile_view.sql`), built by
`python -m src.profile_engine`. Profiles contain observed behaviour and model-estimated propensities only -
no demographics and no psychological labels.

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import pandas as pd
from src.db import connect
from src.profile_engine import VIEW, example_ids, get_profile, render_text

con = connect(read_only=True)
print(con.execute(f"SELECT COUNT(*) AS profiles, MAX(cutoff_date) AS scoring_cutoff FROM {VIEW}").df())

   profiles scoring_cutoff
0      4261     2011-12-10


## 1 · Segment × propensity in SQL - who is likely to buy, by segment

In [2]:
con.execute(f'''
    SELECT segment_name,
           COUNT(*)                                                    AS customers,
           ROUND(AVG(propensity_90d), 3)                               AS avg_propensity_90d,
           ROUND(AVG(CASE WHEN propensity_decile >= 9 THEN 1.0 ELSE 0 END), 3) AS share_top_20pct,
           ROUND(AVG(CASE WHEN propensity_decile <= 3 THEN 1.0 ELSE 0 END), 3) AS share_bottom_30pct,
           ROUND(MEDIAN(gross_spend_365d), 0)                          AS median_spend_365d
    FROM {VIEW}
    GROUP BY segment_name
    ORDER BY avg_propensity_90d DESC
''').df()

,segment_name,customers,avg_propensity_90d,share_top_20pct,share_bottom_30pct,median_spend_365d
0,Core high-value repeat accounts,1071,0.738,0.720,0.001,2740.0
1,Recent seasonal buyers,1354,0.325,0.041,0.250,598.0
2,High-cancellation accounts,115,0.283,0.026,0.478,450.0
3,Lapsed buyers,1234,0.257,0.013,0.452,408.0
4,Low-spend occasional buyers,487,0.209,0.014,0.669,183.0


## 2 · Typical customer per segment (closest to the segment's median propensity)

In [3]:
examples = example_ids(con)
examples

,segment_name,customer_id,propensity_90d,propensity_decile
0,Core high-value repeat accounts,16983,0.747,9
1,Recent seasonal buyers,14240,0.297,5
2,Lapsed buyers,17708,0.224,4
3,High-cancellation accounts,14034,0.217,4
4,Low-spend occasional buyers,15603,0.178,2


## 3 · Full profiles

In [4]:
for cid in examples["customer_id"].head(3):
    print(render_text(get_profile(cid, con)))
    print("\n" + "-" * 64 + "\n")

CUSTOMER 16983  -  profile as of 2011-12-10

L1  Account
    first purchase 2010-09-19 | last purchase 2011-11-27 | customer for 447 days
    market: UK  (descriptive only; not used by any model, only for the fairness audit)

L2  Purchasing behaviour
    orders in the last 12 months               6   [higher than 81% of active customers]
    spend in the last 12 months                £1,742   [higher than 77% of active customers]
    average order value (12 months)            £290   [higher than 50% of active customers]
    products per order                         20.7
    units per order (median)                   197
    distinct products bought                   151   [higher than 80% of active customers]
    product groups bought                      9
    share of spend in main product group       24%
    share of spend on Christmas products       12%
    share of lines re-ordering a product       34%
    price paid vs typical price                0.95
    cancellations per orde

## 4 · The same profile as JSON (what an API or CRM would receive)

In [5]:
print(json.dumps(get_profile(examples["customer_id"].iloc[0], con), indent=2, default=str)[:3000])

{
  "found": true,
  "customer_id": 16983,
  "scoring_cutoff": "2011-12-10",
  "L1_attributes": {
    "first_purchase_date": "2010-09-19",
    "last_purchase_date": "2011-11-27",
    "customer_for": "447 days",
    "market": "UK",
    "market_note": "descriptive only; not used by any model, only for the fairness audit"
  },
  "L2_behaviour": [
    {
      "feature": "frequency_365d",
      "label": "orders in the last 12 months",
      "value": "6",
      "raw": 6.0,
      "context": "higher than 81% of active customers"
    },
    {
      "feature": "gross_spend_365d",
      "label": "spend in the last 12 months",
      "value": "\u00a31,742",
      "raw": 1742.38,
      "context": "higher than 77% of active customers"
    },
    {
      "feature": "aov_365d",
      "label": "average order value (12 months)",
      "value": "\u00a3290",
      "raw": 290.3966666666667,
      "context": "higher than 50% of active customers"
    },
    {
      "feature": "avg_products_per_order",
      "

## 5 · Unknown or inactive customer - handled explicitly

In [6]:
print(render_text(get_profile("99999999", con)))

Customer 99999999 is not among the 4,261 customers with a purchase in the 365 days before the scoring date (2011-12-10). Profiles cover active, identified customers only (anonymous transactions have no customer ID).


In [7]:
con.close()

## Responsible use
- **Propensity is an association, not a cause.** A high score means similar past behaviour was often
  followed by a purchase; it does not say a contact will change anything. Test actions with a randomised
  hold-out group before scaling them.
- **Decile over flag.** The yes/no flag over-selects in peak season (Step 6 findings); targeting uses deciles.
- **Market is descriptive only.** It is never a model input and is used solely for the fairness audit.
- **Accounts, not people.** Many accounts are businesses (wholesale buyers); profiles describe account behaviour.